In [11]:
import sys
import os
import re
import importlib

import torch
import pandas as pd

from data_pipeline import (
    deduplicate,
    SimpleTokenizer,
    tokenize_corpus,
    pack_sequences,
    PreTrainingDataLoader,
    clean_text
)
import mini_gpt_torch
importlib.reload(mini_gpt_torch)
from mini_gpt_torch import MiniGPT, cross_entropy_loss, generate

print("Import Packages from mini_gpt_torch and data_pipline")





Import Packages from mini_gpt_torch and data_pipline


In [15]:
sampel_1 = "i am mohsen mohebbi"
clean_text(sampel_1)
sampel_2 = ("سلام Mohsen! این یک متن تستی است.\nHello World 123\nقیمت محصول: 2500 تومان\nEmail: test@example.com\nPython و Machine Learning خیلی جالب هستند.")
clean_text(sampel_2)

'Mohsen! . Hello World 123 : 2500 Email: test@example.com Python Machine Learning .'

1. def clear_data is NOT for farsi

In [16]:
def clean_persian_text(text: str) -> str:
    """Persian clear text 
    """
    if not isinstance(text, str):
        return ""
    # remove html
    text = re.sub(r"<[^>]+>", " ", text)
    # remove link
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # standart space
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


Load dataset


In [ ]:
print("1. Load dataset".center(60, "-"))
df = pd.read_csv("comments_raw_canonical.csv")
df = df[["specialty", "text"]]
df.dropna(subset=["text"], inplace=True)

raw_docs = []
for _, row in df.iterrows():
    cleaned = clean_persian_text(str(row["text"]))
    if len(cleaned.split()) >= 3:  # comments more than  3 vocabes
        spec = clean_persian_text(str(row["specialty"])) if pd.notna(row["specialty"]) else ""
        doc = f"[{spec}] {cleaned}" if spec else cleaned
        raw_docs.append(doc)

print(f"Count comment is -----  {len(raw_docs):,}")

------------------------Load dataset------------------------
Count comment is -----  45,454


MinHash/LSH
remove dublicate doc

In [ ]:
print("2. MinHash/LSH".center(60, "-"))
sample_size = min(15000, len(raw_docs))
docs_sample = raw_docs[:sample_size]
print(f"\n remove dublicate {sample_size:,} doc")
unique_docs, removed_count = deduplicate(docs_sample, threshold=0.85, num_hashes=64, bands=8)
print(f"count doc unique : {len(unique_docs):,} (count removed comment {removed_count:,})")


------------------------MinHash/LSH-------------------------

 remove dublicate 15,000 doc
count doc unique : 14,620 (count removed comment 380)


Tokenizetr BPE

In [44]:
print("3. Learn SimpleTokenizer".center(60, "-"))

corpus_sample_for_bpe = "\n".join(unique_docs[:3000])
tokenizer = SimpleTokenizer(vocab_size=256)
tokenizer.train_bpe(corpus_sample_for_bpe, num_merges=300)

vocab_size = tokenizer.vocab_size()
print(f"end Vocab Size: {vocab_size}")


------------------3. Learn SimpleTokenizer------------------
end Vocab Size: 557


In [45]:
print("4. change doc to tokens".center(60, "-"))
flat_tokens = tokenize_corpus(unique_docs, tokenizer)

SEQ_LEN = 64
BATCH_SIZE = 16
packed_seqs, att_masks = pack_sequences(flat_tokens, seq_length=SEQ_LEN + 1, pad_id=tokenizer.pad_id)
print(f"Number of sequences ready for training {len(packed_seqs):,}")

train_loader = PreTrainingDataLoader(packed_seqs, att_masks, batch_size=BATCH_SIZE, shuffle=True)


------------------4. change doc to tokens-------------------
Number of sequences ready for training 8,038


create model

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("5. Create model".center(60, "-"))
print(f"Use divece:------- {device}")

model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=SEQ_LEN,
    ff_dim=512
).to(device)

print(f"Count parameters model {model.count_parameters():,}")


----------------------5. Create model-----------------------
Use divece:------- cuda
Count parameters model 870,784


leraning loop

In [48]:
print("6. start learning".center(60, "-"))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
EPOCHS = 2
MAX_STEPS = 1000  # speed test
LOG_EVERY = 50

model.train()
step = 0
stopped = False

for epoch in range(1, EPOCHS + 1):
    for batch_x, _ in train_loader:
        step += 1

        # change to tensor
        batch_tensor = torch.tensor(batch_x, dtype=torch.long, device=device)

        x = batch_tensor[:, :-1].contiguous()
        y = batch_tensor[:, 1:].contiguous()

        logits = model(x)
        loss = cross_entropy_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if step == 1 or step % LOG_EVERY == 0:
            print(f"Epoch {epoch} | Step {step:4d} | Loss: {loss.item():.4f}")

        if step >= MAX_STEPS:
            stopped = True
            break
    if stopped:
        break

print(f"Training completed Final loss---- {loss.item():.4f}")


---------------------6. start learning----------------------
Epoch 1 | Step    1 | Loss: 2.2056
Epoch 1 | Step   50 | Loss: 2.2894
Epoch 1 | Step  100 | Loss: 2.0174
Epoch 1 | Step  150 | Loss: 1.9596
Epoch 1 | Step  200 | Loss: 1.9843
Epoch 1 | Step  250 | Loss: 2.2328
Epoch 1 | Step  300 | Loss: 2.2587
Epoch 1 | Step  350 | Loss: 1.7480
Epoch 1 | Step  400 | Loss: 2.3480
Epoch 1 | Step  450 | Loss: 2.1092
Epoch 1 | Step  500 | Loss: 2.2255
Epoch 2 | Step  550 | Loss: 1.9424
Epoch 2 | Step  600 | Loss: 1.7309
Epoch 2 | Step  650 | Loss: 2.0517
Epoch 2 | Step  700 | Loss: 2.1107
Epoch 2 | Step  750 | Loss: 2.0892
Epoch 2 | Step  800 | Loss: 2.0492
Epoch 2 | Step  850 | Loss: 1.6994
Epoch 2 | Step  900 | Loss: 1.9753
Epoch 2 | Step  950 | Loss: 2.1692
Epoch 2 | Step 1000 | Loss: 2.1378
Training completed Final loss---- 2.1378


test and make text

In [53]:
print("Sampel text to test".center(60, "-"))
model.eval()
# test_prompt = "دکتر بسیار با حوصله"
test_prompt = input("persian text e.g.دکتر بسیار با حوصله   :   ")

prompt_tokens = tokenizer.encode(test_prompt)
print(f"\nPrompt: '{test_prompt}'")
print("Generating the rest of the comment--- ")

output_tokens = generate(model, prompt_tokens, max_new_tokens=40, temperature=0.7)
generated_text = tokenizer.decode(output_tokens)
print(f"\nGenerated Result:\n{generated_text}")

--------------------Sampel text to test---------------------

Prompt: 'دکتر با حوصله بود'
Generating the rest of the comment--- 

Generated Result:
دکتر با حوصله بود[جراح و متخصص زنان زایمان نازایی] دکتر بسیار عالی و باتجربه و فوق العاده ای هستند[جراح و متخصص زنان زایمان نازایی] عالی بودن[جراح و متخصص زنان
